# WideBind Colab Training

**Model**: D=4096, 32 layers, 32 experts, SwiGLU, QK-RMSNorm, RoPE θ=1e6, pos_id binding
**Memory**: VSA multi-scale + cognitive mirror + **Trajectory Manifold** (FCF: beams of transitions, Zeckendorf decay, φ-закон: 0 лучей = ceil(√buffer), большой буфер 1024 → 32 луча)
**AMP safe**: все einsum заменены на matmul/element-wise на горячем пути — fp16-CUBLAS не падает (T4/L4)
**VRAM target**: T4 (16GB) or better
**Data**: token_stream_*.bin files in Google Drive

---


In [ ]:
# @title 1. Mount Drive & Install Deps
import os, sys, math, time, glob, json
# Борьба с фрагментацией CUDA-аллокатора (T4 16GB, 32 слоя D=4096)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from google.colab import drive
drive.mount('/content/drive')

# Config — point this to your Drive folder with widebind/ and data/
DRIVE_ROOT = '/content/drive/MyDrive/widebind'  # @param {type:'string'}
DATA_DIR    = os.path.join(DRIVE_ROOT, 'data')
SAVE_DIR    = os.path.join(DRIVE_ROOT, 'checkpoints')
LOG_DIR     = os.path.join(DRIVE_ROOT, 'logs')

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'DRIVE_ROOT={DRIVE_ROOT}')
print(f'DATA_DIR={DATA_DIR}')
print(f'SAVE_DIR={SAVE_DIR}')


In [ ]:
# @title 2. Clone Code from GitHub (data on Drive)
import subprocess

DST = '/content/widebind'
if not os.path.exists(DST):
    print('Cloning from GitHub...')
    subprocess.run(['git', 'clone', 'https://github.com/BlackCatSpb/widebind.git', DST], check=True)
    print('Done.')
else:
    print('Already cloned, pulling latest...')
    subprocess.run(['git', '-C', DST, 'pull'], check=True)

sys.path.insert(0, DST)
os.chdir(DST)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# @title 3. Verify GPU & Imports
import torch
import torch.nn.functional as F
import numpy as np
from torch.serialization import add_safe_globals
from core import WideBindConfig, WideBindStack, MirrorLRScheduler

add_safe_globals([WideBindConfig])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'N/A'
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if device == 'cuda' else 0
print(f'Device: {device}  GPU: {gpu_name}  VRAM: {gpu_mem:.1f} GB')
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')

In [ ]:
# @title 4. Build Model (big: D=4096, 32 layers)cfg = WideBindConfig(    D=4096,    n_layers=16,  # 32 requires 24GB+ VRAM; 16 is comfortable on T4 16GB    orth_weight=0.0,  # ortho gram off by default (memory)    bind_K=64,    mlp_groups=32,    mlp_expand=4,    seq_len=512,    lr=3e-4,    max_steps=300000,    warmup_steps=2000,    log_interval=100,    eval_interval=1000,    save_interval=5000,    scheduler='mirror',    per_layer_ls_lr=True,  # per-layer LR modulation from fast/slow EMA var(log_scale)    ls_ema_fast=0.99,    ls_ema_slow=0.999,    ls_mult_min=0.5,    ls_mult_max=2.0,    ls_mirror_mult_max=2.0,    private_mem=True,    expert_asymmetry=True,    meta_trust=True,    data_dir=DATA_DIR,    save_dir=SAVE_DIR,    log_dir=LOG_DIR,    grad_clip=0.5,    conv_kernel=48,    traj_manifold=True,    traj_beams=0,    traj_buffer_size=1024,    traj_gain=0.05,)model = WideBindStack(cfg).to(device)n_params = model.param_count()print(f'Model: {n_params:,} params ({n_params/1e6:.2f}M)')

In [ ]:
# @title 5. Auto Batch Size & Mixed Precision
def find_best_batch_size(model, seq_len, device, start=2):
    lo, hi = 1, start
    best = 1
    torch.cuda.empty_cache()
    try:
        x = torch.randint(0, 50000, (start, seq_len), device=device)
        h = model.embed_tokens(x)
        out, _, _, _ = model(h, None)
        out[:, :1].sum().backward()
        best = start
    except RuntimeError:
        hi = start // 2
    finally:
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
    while lo <= hi:
        mid = (lo + hi) // 2
        torch.cuda.empty_cache()
        try:
            x = torch.randint(0, 50000, (mid, seq_len), device=device)
            h = model.embed_tokens(x)
            out, _, _, _ = model(h, None)
            out[:, :1].sum().backward()
            best = mid
            lo = mid + 1
        except RuntimeError:
            hi = mid - 1
        finally:
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    torch.cuda.empty_cache()
    return max(1, best)

cfg.batch_size = find_best_batch_size(model, cfg.seq_len, device, start=2)
print(f'Optimal batch size: {cfg.batch_size}')
print(f'  Tokens/step: {cfg.batch_size * cfg.seq_len}')

# Автодетект поддержки fp16 GEMM: K80/P100 и некоторые сессии Colab его не
# имеют — cublas fp16-тензорные ядра просто отсутствуют (CUBLAS_STATUS_NOT_SUPPORTED).
def probe_fp16_gemm():
    if device != 'cuda':
        return False
    try:
        a = torch.randn(64, 64, device='cuda', dtype=torch.float16)
        b = a @ a
        torch.cuda.synchronize()
        del a, b
        torch.cuda.empty_cache()
        return True
    except Exception:
        return False

FP16_OK = probe_fp16_gemm()
print(f'GPU: {gpu_name} | fp16 GEMM supported: {FP16_OK}')

if FP16_OK:
    scaler = torch.amp.GradScaler('cuda')
    print('  Using fp16 mixed precision + GradScaler')
else:
    scaler = None
    print('  fp16 недоступен на этой GPU → тренировка в fp32 (без autocast)')


In [ ]:
# @title 6. Optimizer & LR Scheduler
param_groups = model.param_groups()
optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.95))

scheduler = MirrorLRScheduler(model, optimizer, cfg.lr,
    warmup=cfg.warmup_steps, target_var=cfg.target_var,
    mag_threshold=cfg.mag_threshold, lr_min_ratio=cfg.lr_min_ratio,
    max_decay_steps=cfg.max_decay_steps,
    var_min_for_lr_decay=cfg.var_min_for_lr_decay,
    cfg=cfg)
print('Scheduler: MirrorLRScheduler')

In [ ]:
# @title 7. Data Streams
class TokenStream:
    def __init__(self, path):
        self.data = np.memmap(path, dtype=np.uint16, mode='r')
        self.len = len(self.data)
    def get_batch(self, seq_len, batch_size, offset, vocab=50000):
        needed = batch_size * seq_len + 1
        if offset + needed > self.len:
            offset = 0
        chunk = self.data[offset:offset + needed]
        if vocab is not None:
            chunk = np.clip(chunk, 0, vocab - 1)
        x = torch.from_numpy(chunk[:batch_size * seq_len].reshape(batch_size, seq_len).copy())
        y = torch.from_numpy(chunk[1:batch_size * seq_len + 1].reshape(batch_size, seq_len).copy())
        return x.long(), y.long(), offset + batch_size * seq_len

stream_files = sorted(glob.glob(os.path.join(DATA_DIR, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DATA_DIR, 'token_stream_*.bin')))
if not stream_files:
    print('WARNING: No token_stream_*.bin found!')
    print(f'  Looked in: {DATA_DIR}')
    print('  Using random data for testing')
    streams = []
else:
    streams = [TokenStream(f) for f in stream_files]
    total_tokens = sum(s.len for s in streams)
    print(f'Found {len(streams)} files, {total_tokens:,} total tokens')

In [ ]:
# @title 8. Resume Checkpoint (optional)
start_step = 0
state = None
best_val_loss = float('inf')

ckpt_files = sorted(
    glob.glob(os.path.join(SAVE_DIR, 'step_*.pt')),
    key=lambda p: int(os.path.basename(p).split('_')[1].split('.')[0]))

if not ckpt_files:
    # Also check for best.pt
    best_ckpt = os.path.join(SAVE_DIR, 'best.pt')
    if os.path.exists(best_ckpt):
        ckpt_files = [best_ckpt]

if ckpt_files:
    latest = ckpt_files[-1]
    print(f'Resuming from {latest}')
    ckpt = torch.load(latest, map_location='cpu', weights_only=True)  # FIX: load to CPU, avoid GPU OOM transient
    from core.migrate import migrate_state_dict
    sd, n_mig = migrate_state_dict(dict(ckpt['model']), model)
    if n_mig:
        print(f'  MIGRATED {n_mig} keys (W_out +K, bind_coh_gate=0, freq_scale=1.0)')
    miss, unex = model.load_state_dict(sd, strict=False)
    if miss:
        print(f'  Missing keys: {len(miss)}')
    if unex:
        print(f'  Unexpected keys: {len(unex)}')
    def _restore_optimizer(optimizer, model, ckpt_opt):
        old_names = ckpt_opt.get('param_names') if isinstance(ckpt_opt, dict) else None
        if old_names is None:
            print('  WARNING: no param_names - optimizer state NOT restored (fresh Adam)')
            return False
        names = {id(p): n for n, p in model.named_parameters()}
        pos = {id(p): i for i, p in enumerate(
            (p for g in optimizer.param_groups for p in g['params']))}
        new_sd = optimizer.state_dict()
        new_sd['state'] = {}
        old_state = ckpt_opt.get('state', {})
        old_groups = ckpt_opt.get('param_groups', [])
        moved = skipped = 0
        for name, p in model.named_parameters():
            if name not in old_names:
                skipped += 1
                continue
            si = old_names.index(name)
            st = old_state.get(str(si)) if str(si) in old_state else old_state.get(si)
            if st is None:
                continue
            if tuple(st['exp_avg'].shape) != tuple(p.shape):
                if (name.endswith('bind.W_out') and len(st['exp_avg'].shape) == 2
                        and st['exp_avg'].shape[1] == p.shape[1]
                        and st['exp_avg'].shape[0] < p.shape[0]):
                    st = {k: (v[:p.shape[0]] if isinstance(v, torch.Tensor) and v.dim() == 2 else v)
                          for k, v in st.items()}
                    moved += 1
                else:
                    skipped += 1
                    continue
            else:
                moved += 1
            new_sd['state'][pos[id(p)]] = {k: (v.clone() if isinstance(v, torch.Tensor) else v)
                                             for k, v in st.items()}
        for gi in range(min(len(new_sd['param_groups']), len(old_groups))):
            if 'lr' in old_groups[gi]:
                new_sd['param_groups'][gi]['lr'] = old_groups[gi]['lr']
        optimizer.load_state_dict(new_sd)
        print(f'  Optimizer restored by name: {moved} slots, {skipped} skipped')
        return moved > 0
    ckpt['optimizer'] = None  # FIX: skip optimizer state -> fresh Adam (no OOM on 15GB)
    if not _restore_optimizer(optimizer, model, ckpt['optimizer']):
        print('  WARNING: optimizer state not restored (fresh Adam)')
    if 'scheduler' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler'])
    start_step = ckpt.get('step', 0)
    reasoning_enabled_step = ckpt.get('reasoning_enabled_step', 0)
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'  Resumed at step {start_step}')
else:
    reasoning_enabled_step = 0
    print('No checkpoint found, starting fresh')

In [ ]:
import osos.environ['CUDA_LAUNCH_BLOCKING'] = '1'# @title 9. 🚀 TRAINING LOOP (auto-fallback на OOM)import torch._dynamotorch._dynamo.config.suppress_errors = Truestream_idx = 0offset = 0tokens_seen = 0t0 = time.time()rng = torch.Generator(device='cpu').manual_seed(42)print(f'Training: step {start_step} -> {cfg.max_steps}')print(f'  ({cfg.max_steps - start_step} steps remaining)\n')try:    for step in range(start_step, cfg.max_steps):        model.train()        retry = True        while retry:            retry = False            try:                # ── Data ──                if streams:                    if offset == 0:                        stream_idx = torch.randint(0, len(streams), (1,), generator=rng).item()                        state = None                    x, y, offset = streams[stream_idx].get_batch(cfg.seq_len, cfg.batch_size, offset, cfg.vocab)                else:                    x = torch.randint(0, cfg.vocab, (cfg.batch_size, cfg.seq_len))                    y = torch.randint(0, cfg.vocab, (cfg.batch_size, cfg.seq_len))                x, y = x.to(device), y.to(device)                # ── Forward ──                with torch.amp.autocast(device_type='cuda', enabled=scaler is not None):                    h = model.embed_tokens(x)                    out, state, _, _ = model(h, state, step=step)                    ce_loss, aux_dict = model.compute_losses(out, y, h_emb=h)                    loss = ce_loss + sum(aux_dict.values())                # ── Backward ──                if scaler:                    scaler.scale(loss).backward()                else:                    loss.backward()                tokens_seen += cfg.batch_size * cfg.seq_len                # Per-layer LS-based LR modulation (cfg.per_layer_ls_lr)                ls_mults = getattr(scheduler, '_ls_mult', None)                if ls_mults is not None:                    for i, layer in enumerate(model.layers):                        ls_m = ls_mults[i]                        for p in layer.base_parameters:                            if p.grad is not None:                                p.grad.mul_(ls_m)                        for p in layer.mirror_parameters:                            if p.grad is not None:                                p.grad.mul_(ls_m)                # ── Optimizer step ──                if scaler:                    if cfg.grad_clip > 0:                        scaler.unscale_(optimizer)                        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)                    scaler.step(optimizer)                    scaler.update()                else:                    if cfg.grad_clip > 0:                        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)                    optimizer.step()                optimizer.zero_grad(set_to_none=True)                scheduler.step()                # ── Detach state ──                if state is not None:                    state = tuple(tuple(t.detach() if isinstance(t, torch.Tensor) else t for t in s) if s else None for s in state)            except torch.cuda.OutOfMemoryError:                torch.cuda.empty_cache()                if cfg.seq_len > 64:                    cfg.seq_len //= 2                elif cfg.batch_size > 1:                    cfg.batch_size //= 2                else:                    raise                state = None                offset = 0                retry = True                print(f'  [OOM] retry: seq_len={cfg.seq_len} batch={cfg.batch_size}')        # ── Log ──        if step % cfg.log_interval == 0:            dt = time.time() - t0            tok_s = tokens_seen / max(dt, 1e-8)            lr = scheduler.get_last_lr()[0]            mem_gb = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else 0            aux_str = ' '.join(f'{k}={v:.4f}' for k, v in sorted(aux_dict.items()) if abs(v) > 1e-6)            print(f'step={step:>6}  loss={loss.item():.4f}  ce={ce_loss.item():.4f}  '                  f'lr={lr:.2e}  tok/s={tok_s:.0f}  mem={mem_gb:.1f}GB')            if aux_str:                print(f'  aux: {aux_str}')            if device == 'cuda':                torch.cuda.reset_peak_memory_stats()        # ── Eval ──        if step > 0 and step % cfg.eval_interval == 0:            model.eval()            val_loss = 0.0            n_val = 0            with torch.no_grad():                for s in streams[:3]:                    voff = max(s.len // 4, cfg.batch_size * cfg.seq_len + 1)                    for _ in range(min(100, s.len // (cfg.batch_size * cfg.seq_len))):                        vx, vy, voff = s.get_batch(cfg.seq_len, cfg.batch_size, voff, cfg.vocab)                        if voff == 0:                            break                        vx, vy = vx.to(device), vy.to(device)                        h = model.embed_tokens(vx)                        out, _, _, _ = model(h, None, adaptive=False)                        ce, _ = model.compute_losses(out, vy, h_emb=h)                        val_loss += ce.item()                        n_val += 1            if n_val > 0:                val_loss /= n_val                val_ppl = math.exp(min(val_loss, 20))                print(f'  EVAL step={step}: val_loss={val_loss:.4f} val_ppl={val_ppl:.2f}')                scheduler.report_val_loss(val_loss)                if val_loss < best_val_loss:                    best_val_loss = val_loss                    torch.save({                        'step': step, 'model': model.state_dict(),                        'optimizer': optimizer.state_dict(),            'param_names': [next(n for n, pp in model.named_parameters() if id(pp) == id(p)) for g in optimizer.param_groups for p in g['params']],                        'scheduler': scheduler.state_dict(),                        'best_val_loss': best_val_loss, 'cfg': cfg,                    }, os.path.join(SAVE_DIR, 'best.pt'))                    print(f'  Saved best to best.pt')            model.train()            if device == 'cuda':                torch.cuda.empty_cache()        # ── Save ──        if step > 0 and step % cfg.save_interval == 0:            save_path = os.path.join(SAVE_DIR, f'step_{step}.pt')            torch.save({                'step': step, 'model': model.state_dict(),                'optimizer': optimizer.state_dict(),            'param_names': [next(n for n, pp in model.named_parameters() if id(pp) == id(p)) for g in optimizer.param_groups for p in g['params']],                'scheduler': scheduler.state_dict(),                'best_val_loss': best_val_loss, 'cfg': cfg,            }, save_path)            print(f'  Saved checkpoint to {save_path}')except KeyboardInterrupt:    print('\nInterrupted — saving...')    save_path = os.path.join(SAVE_DIR, f'interrupt_step_{step}.pt')    torch.save({        'step': step, 'model': model.state_dict(),        'optimizer': optimizer.state_dict(),            'param_names': [next(n for n, pp in model.named_parameters() if id(pp) == id(p)) for g in optimizer.param_groups for p in g['params']],        'scheduler': scheduler.state_dict(),        'best_val_loss': best_val_loss, 'cfg': cfg,    }, save_path)    print(f'Saved to {save_path}')print('\nTraining complete!')